## Import

In [ ]:
import os
import gc
import sys
import warnings
import numpy as np
import pandas as pd
from glob import glob

sys.path.append("../input/pythonbox")
from box import Box

# Torch
import torch
import torch.nn as nn
import torchvision.transforms as T
from torchvision.io import read_image
from torch.utils.data import DataLoader, Dataset 
from torchvision.models import efficientnet_v2_l

# Lightning
import pytorch_lightning as pl
from pytorch_lightning import LightningDataModule, LightningModule, seed_everything

In [ ]:
!pip uninstall timm --yes

sys.path.append("/kaggle/input/timmmaster")
# import timm
from timm import create_model

# print(timm.__version__)

## Config

In [ ]:
config = {'exp_name':'exp_016',
          'root': '../input/petfinder-pawpularity-score/',  # Data root
          'seed': 2023,
          'n_splits': 5,
          'image_size': 384,
          'model':{
              'package': 'timm',  # timm or torchvision
              'name': 'cait_m36_384',
              'output_dim': 1,
              'pretrain': False,
          },
          'save_dir': 'exp016',  # 儲存權重與log的資料夾
          'test_loader': {
              'batch_size': 32,
              'shuffle': False,
              'num_workers': os.cpu_count(),
              'pin_memory': False,
              'drop_last': False
          },
}

config = Box(config)

## Fix Seed

In [ ]:
seed_everything(config.seed)

## Tools

In [ ]:
def RMSE(predict,target):
    return torch.sqrt(nn.MSELoss()(predict.float(), target.float()))

## Dataset

In [ ]:
class PetfinderDataset(Dataset):
    """Dataset
    Args:
        df: the dataframe from csv, and the "Id" column needs to be the path of Image
    """
    def __init__(self, df, transform=None, image_size=224):
        
        self._X = df["Id"].values
        self._y = None
        self.transform = transform
        
        # 判斷有沒有分數
        if "Pawpularity" in df.keys():
            self._y = df["Pawpularity"].values
            
    def __len__(self):
        return len(self._X)

    def __getitem__(self, idx):
        image_path = self._X[idx]
        image = read_image(image_path)
        image = self.transform(image)
        
        if self._y is not None:
            label = self._y[idx]
            return image, label
        return image

## Model

In [ ]:
class Model(pl.LightningModule):
    def __init__(self, hparams):
        super().__init__()
        self.save_hyperparameters(hparams)  # 儲存超參數
        
        self.__build_model()
        
    def __build_model(self):
        if self.hparams.model.package == 'timm':
            self.backbone = create_model(self.hparams.model.name,
                                         pretrained=False, 
                                         num_classes=0, 
                                         in_chans=3)
            num_features = self.backbone.num_features
        
        elif self.hparams.model.package == 'torchvision':
            # weights = 'DEFAULT' if self.hparams.model.pretrain else None
            self.backbone = eval(self.hparams.model.name)(weights=None)
            num_features = 1000
            
        self.fc = nn.Sequential(nn.Dropout(0.5), 
                                nn.Linear(num_features, 
                                          self.hparams.model.output_dim))

    def forward(self, x):
        f = self.backbone(x)
        out = self.fc(f)
        return out

    def predict_step(self, batch, batch_idx):
        images = batch
        pred = self(images).sigmoid()  # return後會自動轉成numpy
        if pred.dim == 3:
            pred = pred.squeeze()
        return pred


## Test

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]  # RGB
IMAGENET_STD = [0.229, 0.224, 0.225]  # RGB

test_transform = T.Compose([T.Resize(config.image_size),
                            T.CenterCrop([config.image_size, config.image_size]),
                            T.ConvertImageDtype(torch.float),
                            T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)])

stage = 'test'
df = pd.read_csv(os.path.join(config.root, stage+'.csv'))
df["Id"] = df["Id"].apply(lambda x: os.path.join(config.root, stage, x + ".jpg")) # 將ID改成圖片路徑
if "Pawpularity" in df.keys():
    test_y = df["Pawpularity"].astype(float).apply(lambda x: x / 100.).to_numpy() # 將Pawpularity軟換到[0, 1]
    df.drop('Pawpularity', axis=1, inplace=True)

# df to dataset
predict_data = PetfinderDataset(df, test_transform, config.image_size)
predict_loader = DataLoader(predict_data, **config.test_loader)

### kFold predictions

In [ ]:
warnings.filterwarnings("ignore")  # 關閉 Warning
torch.set_float32_matmul_precision('high')  # 設置高精度(根據顯卡調整)

total_predictions = []
for fold in range(config.n_splits):
    model_weight = glob(f'/kaggle/input/{config.save_dir}/fold_{fold}/version_0/*.ckpt')[0]

    model = Model(config).load_from_checkpoint(model_weight, map_location='cuda:0')

    trainer = pl.Trainer(logger=False, precision='16-mixed')
    predictions = trainer.predict(model, dataloaders=predict_loader)
    
    total_predictions.append(np.concatenate(predictions).flatten())
    
    # 清除變數與快取
    del model, trainer, predictions
    torch.cuda.empty_cache()
    gc.collect()

### 計算平均預測分數

In [ ]:
mean_predictions = np.array(total_predictions).mean(axis=0)
print(mean_predictions)

### 計算RMSE

In [ ]:
# RMSE(torch.tensor(mean_predictions), torch.tensor(test_y)).item()

### 儲存平均預測分數

In [ ]:
# 儲存csv
df = pd.read_csv(os.path.join(config.root, stage+'.csv'))
df_id = df['Id']

mean_predicts_df = pd.DataFrame(mean_predictions * 100., columns=["Pawpularity"])
df = pd.concat([df_id, mean_predicts_df], axis=1)
df.to_csv("submission.csv", index=False)
df.head(5)